# <font color='blue'> Chapter 24: Transfer Learning </font>

Training a deep Convolutional Neural Network (CNN) from scratch often requires

- millions of training images,
- powerful GPUs,
- days or even weeks of computation.

For many practical applications, such resources are unavailable.

Fortunately, neural networks trained on one task often learn representations that are useful for many other tasks.

This observation forms the basis of **Transfer Learning**, one of the most important techniques in modern Deep Learning.

Instead of training a new model from the beginning, we start with a **pretrained model**, reuse the features it has already learned, and adapt it to a new problem.

Transfer Learning dramatically reduces training time, improves accuracy on small datasets, and has become the standard approach in computer vision.

---

# <font color='orange'> 1. Motivation </font>

Suppose we wish to build a CNN that recognizes different species of flowers.

Assume we possess only

2,000

labelled images.

Training a deep CNN from scratch using such a small dataset would likely result in severe overfitting.

Instead,

we begin with a CNN already trained on

ImageNet,

which contains more than

14 million

images covering

1,000

object categories.

Rather than learning edges, textures, and shapes from scratch,

our model reuses these previously learned features.

---

# <font color='orange'> 2. The Idea Behind Transfer Learning </font>

Consider two tasks.

Task A

```
Recognize

Cars

Dogs

Cats

Birds

...
```

Task B

```
Recognize

Tulips

Roses

Sunflowers
```

Although the final classification differs,

both tasks require detection of

- edges,
- textures,
- shapes,
- colours.

The lower layers of a CNN therefore remain useful.

Only the final classifier needs to be adapted.

---

# <font color='orange'> 3. Why Does Transfer Learning Work? </font>

Earlier CNN layers learn

```
Edges

↓

Corners

↓

Textures
```

These features are almost universal.

Deeper layers learn

```
Object Parts

↓

Specific Objects
```

Only the final layers become highly task-specific.

Consequently,

we often keep the early layers fixed

and retrain only the final classification layers.

---

# <font color='orange'> 4. Feature Extraction </font>

The simplest form of Transfer Learning is

**Feature Extraction**.

The pretrained CNN acts as a fixed feature extractor.

```
Input Image

↓

Pretrained CNN

↓

Feature Vector

↓

New Dense Layer

↓

Prediction
```

The pretrained weights remain unchanged.

Only the new classifier is trained.

---

# <font color='orange'> 5. Fine-Tuning </font>

A more advanced approach is

**Fine-Tuning**.

Instead of freezing every pretrained layer,

we unfreeze some of the deeper layers.

```
Early Layers

Frozen

↓

Later Layers

Trainable

↓

New Classifier

Trainable
```

Fine-tuning allows the network to adapt its learned features to the new dataset.

---

# <font color='orange'> 6. Freezing Layers </font>

In Keras,

layers can be frozen using

```python
base_model.trainable = False
```

During backpropagation,

the frozen layers

- participate in forward propagation,
- compute activations,

but

their weights are **not updated**.

Only trainable layers receive gradient updates.

---

# <font color='orange'> 7. Unfreezing Layers </font>

Later,

we may unfreeze selected layers.

```python
base_model.trainable = True
```

Often,

only the final convolutional blocks are fine-tuned.

A smaller learning rate is typically used during fine-tuning,

preventing large changes to the pretrained weights.

---

# <font color='orange'> 8. Popular Pretrained CNN Architectures </font>

Several pretrained models are available in Keras.

| Model | Characteristics |
|:---|:---|
| VGG16 | Simple and deep, but many parameters |
| VGG19 | Slightly deeper than VGG16 |
| ResNet50 | Residual connections, excellent accuracy |
| InceptionV3 | Multi-scale convolutions |
| MobileNetV2 | Lightweight, suitable for mobile devices |
| EfficientNet | Excellent accuracy with relatively few parameters |

These models are pretrained on the ImageNet dataset and can be loaded directly using `keras.applications`.

---

# <font color='orange'> 9. Loading a Pretrained Model </font>

Example

```python
from tensorflow.keras.applications import ResNet50

base_model = ResNet50(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)
```

Here,

- `weights="imagenet"` loads pretrained weights,
- `include_top=False` removes the original classifier,
- `input_shape` specifies the dimensions of the new input images.

---

# <font color='orange'> 10. Adding a New Classifier </font>

The pretrained feature extractor is combined with a new classifier.

```python
model = keras.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        256,
        activation="relu"
    ),

    layers.Dense(
        num_classes,
        activation="softmax"
    )

])
```

Only the final Dense layers are specific to the new classification task.

---

# <font color='orange'> 11. Training Workflow </font>

A typical Transfer Learning pipeline is

```
Load Pretrained Model

↓

Freeze Layers

↓

Train New Classifier

↓

Unfreeze Final Layers

↓

Fine-Tune

↓

Evaluate
```

This two-stage approach often provides better performance than training everything simultaneously.

---

# <font color='orange'> 12. Advantages of Transfer Learning </font>

Transfer Learning

- requires less data,
- trains much faster,
- often achieves higher accuracy,
- reduces overfitting,
- leverages knowledge learned from large datasets.

These advantages make it the preferred approach for many practical computer vision tasks.

---

# <font color='orange'> 13. Limitations </font>

Transfer Learning is not always appropriate.

If the source and target tasks differ substantially,

the pretrained features may be less useful.

For example,

a model trained on natural photographs may not transfer effectively to

- medical imaging,
- satellite radar,
- microscopic cell images,

without careful fine-tuning.

---

# <font color='orange'> 14. Feature Extraction vs Fine-Tuning </font>

| Feature Extraction | Fine-Tuning |
|:---|:---|
| Pretrained weights fixed | Some pretrained weights updated |
| Faster training | Slower training |
| Lower risk of overfitting | Higher flexibility |
| Small datasets | Larger datasets |
| Lower computational cost | Higher computational cost |

The choice depends on the size of the dataset and its similarity to the original training data.

---

# <font color='red'> 15. Mathematical Foundations </font>

Suppose a pretrained CNN computes

$$
\mathbf{h}
=
f(\mathbf{x};\theta_{\rm pre}),
$$

where

- $\mathbf{x}$ is the input image,
- $\theta_{\rm pre}$ denotes the pretrained parameters,
- $\mathbf{h}$ is the extracted feature representation.

A new classifier then computes

$$
\hat{\mathbf{y}}
=
g(\mathbf{h};\theta_{\rm new}),
$$

where

$\theta_{\rm new}$

are newly initialized trainable parameters.

During **feature extraction**,

only

$$
\theta_{\rm new}
$$

is optimized.

During **fine-tuning**,

both

$$
\theta_{\rm pre}
$$

and

$$
\theta_{\rm new}
$$

are optimized,

typically using a smaller learning rate for the pretrained parameters.

---

# <font color='orange'> 16. Common Misconceptions </font>

### Misconception 1

> Transfer Learning means copying another model without modification.

**False.**

Transfer Learning reuses learned representations, but the model is usually adapted to the new task by replacing or fine-tuning the classification layers.

---

### Misconception 2

> All pretrained layers should always be trainable.

**False.**

Freezing early layers often improves performance on small datasets and reduces the risk of overfitting.

---

### Misconception 3

> Transfer Learning always outperforms training from scratch.

**False.**

When abundant labelled data are available and the target task differs substantially from the source task, training from scratch may perform as well as or better than transfer learning.

---

# <font color='purple'> 17. Conceptual Summary </font>

| Concept | Description |
|:---|:---|
| Transfer Learning | Reusing a pretrained model for a new task |
| Pretrained Model | Model previously trained on a large dataset |
| Feature Extraction | Using a pretrained network with frozen weights |
| Fine-Tuning | Updating some pretrained layers on a new dataset |
| Frozen Layer | Participates in inference but its weights are not updated |
| ImageNet | Large-scale dataset commonly used for pretraining |
| Global Average Pooling | Reduces feature maps before classification |

> **Key Insight:** Transfer Learning enables deep neural networks to reuse knowledge acquired from large datasets, dramatically reducing the amount of data and computation required for new tasks. By freezing early layers and adapting only the later layers or classifier, pretrained CNNs can achieve excellent performance even when only limited training data are available. This approach has become the standard workflow for modern computer vision applications.